# K-Medoid DTW Clustering

**Workflow**
1. Load raw + z-norm data
2. k-search on a fast sample — find the best number of clusters using three metrics
3. Full clustering with the chosen k
4. Validation: Silhouette, Davies-Bouldin, Jaccard stability
5. Plots + export

**Key design choice:** the DTW distance matrix is built once on a sample (~600 series) for the
k-search (fast), then rebuilt on the full dataset for the final clustering (slow but thorough).


## 0. Installs

In [ ]:
# !pip install dtaidistance

## 1. Imports

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

try:
    from dtaidistance import dtw as dtaidtw
    HAS_DTAI = True
except ImportError:
    HAS_DTAI = False
    warnings.warn("pip install dtaidistance")

warnings.filterwarnings("ignore")
print(f"dtaidistance: {HAS_DTAI}")

## 2. Config

Only edit this cell.

In [ ]:
RAW_PATH   = Path("../data/raw/sample_23.csv")
ZNORM_PATH = Path("../data/processed/sample_23_znorm.parquet")
OUT_DIR    = Path("outputs/kmedoid")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DTW_WINDOW   = 30          # Sakoe-Chiba band (days); 30 = ±1 month warping

# k-search range
K_MIN, K_MAX = 2, 12

# Sample size for k-search (fast); None = use all
KSEARCH_SAMPLE = 600

# Final clustering uses all series (set to int to limit for testing)
FINAL_SAMPLE = None

# Consensus settings
CONSENSUS_RUNS  = 8
CONSENSUS_RATIO = 0.85
KMEDOID_ITERS   = 60

print("Config OK")

## 3. Load data

In [ ]:
def load(path):
    path = Path(path)
    df = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    ids    = df.iloc[:, 0].to_numpy()
    values = df.iloc[:, 1:].to_numpy(dtype=np.float64)
    return ids, values

ids_raw,   X_raw   = load(RAW_PATH)
ids_znorm, X_znorm = load(ZNORM_PATH)

print(f"Raw   : {X_raw.shape}")
print(f"Z-norm: {X_znorm.shape}")

## 4. Normalise

Raw data is z-normalised per series.
Z-norm parquet is already normalised — verify and use as-is.
Flat/near-zero series are tagged and excluded from clustering
(they collapse into a degenerate cluster if included), but kept for the final label assignment.


In [ ]:
def z_norm(X):
    mean = X.mean(axis=1, keepdims=True)
    std  = X.std(axis=1, keepdims=True) + 1e-6
    return (X - mean) / std

def check_norm(X, name):
    print(f"{name}: row means={X.mean(axis=1).mean():.3f}  row stds={X.std(axis=1).mean():.3f}")

def active_mask(X, threshold=0.01):
    return X.std(axis=1) > threshold

Z_raw   = z_norm(X_raw)
Z_znorm = X_znorm.copy()   # already normalised

check_norm(Z_raw,   "raw normalised")
check_norm(Z_znorm, "znorm file    ")

mask_raw   = active_mask(Z_raw)
mask_znorm = active_mask(Z_znorm)

print(f"\nFlat series excluded from clustering:")
print(f"  raw   : {(~mask_raw).sum()} / {len(mask_raw)}")
print(f"  znorm : {(~mask_znorm).sum()} / {len(mask_znorm)}")

## 5. DTW distance matrix

In [ ]:
def build_dtw_matrix(Z, window=DTW_WINDOW):
    n = len(Z)
    pairs = n * (n - 1) // 2
    mem_mb = n * n * 8 / 1e6
    print(f"  N={n}  pairs={pairs:,}  matrix≈{mem_mb:.0f} MB", end="  ")
    t0 = time.time()
    raw = dtaidtw.distance_matrix_fast(Z, window=window)
    elapsed = time.time() - t0
    print(f"{elapsed:.1f}s")
    if hasattr(raw, "toarray"):
        D = raw.toarray()
        D = D + D.T
        np.fill_diagonal(D, 0.0)
    else:
        D = np.array(raw)
    return D

## 6. K-medoids + consensus

In [ ]:
def kmedoids(D, k, rng, max_iter=KMEDOID_ITERS):
    n = D.shape[0]
    medoids = rng.choice(n, size=k, replace=False)
    for _ in range(max_iter):
        labels = D[:, medoids].argmin(axis=1)
        new_m  = medoids.copy()
        for c in range(k):
            members = np.where(labels == c)[0]
            if len(members) == 0:
                cands = np.setdiff1d(np.arange(n), new_m)
                if len(cands):
                    new_m[c] = rng.choice(cands)
                continue
            sub     = D[np.ix_(members, members)]
            new_m[c] = members[sub.sum(axis=1).argmin()]
        if np.array_equal(new_m, medoids):
            break
        medoids = new_m
    labels = D[:, medoids].argmin(axis=1)
    return medoids, labels


def consensus(D, k, n_runs=CONSENSUS_RUNS, ratio=CONSENSUS_RATIO, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n   = D.shape[0]
    agree  = np.zeros((n, n))
    cooc   = np.zeros((n, n))
    for _ in range(n_runs):
        sz  = max(k + 1, int(ratio * n))
        idx = np.sort(rng.choice(n, size=sz, replace=False))
        sub = D[np.ix_(idx, idx)]
        _, lbl = kmedoids(sub, k, rng)
        for i, gi in enumerate(idx):
            for j, gj in enumerate(idx):
                cooc[gi, gj]  += 1
                if lbl[i] == lbl[j]:
                    agree[gi, gj] += 1
    sim = np.divide(agree, cooc, out=np.zeros_like(agree), where=cooc > 0)
    np.fill_diagonal(sim, 1.0)
    return sim


def cluster_from_sim(sim, k):
    dist = 1.0 - sim
    np.fill_diagonal(dist, 0.0)
    agg = AgglomerativeClustering(n_clusters=k, metric="precomputed", linkage="average")
    return agg.fit_predict(dist)


def medoid_positions(D, labels, k):
    pos = []
    for c in range(k):
        members = np.where(labels == c)[0]
        sub = D[np.ix_(members, members)]
        pos.append(members[sub.sum(axis=1).argmin()])
    return pos

## 7. Validation metrics (Silhouette, Davies-Bouldin, Jaccard)

In [ ]:
def sil(Z, labels, max_sample=5000, seed=RANDOM_STATE):
    u = np.unique(labels)
    if len(u) < 2 or len(u) >= len(Z):
        return np.nan
    return silhouette_score(Z, labels,
                            sample_size=min(max_sample, len(Z)),
                            random_state=seed)


def dbi(Z, labels):
    u = np.unique(labels)
    if len(u) < 2:
        return np.nan
    return davies_bouldin_score(Z, labels)


def jaccard_stability(Z, labels, k, n_boot=10, ratio=0.8, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    n   = len(Z)
    # Use cluster centroids as proxies for assignment
    centroids = np.array([Z[labels == c].mean(axis=0) for c in range(k)])

    scores = np.zeros((n_boot, k))
    for run in range(n_boot):
        sz  = max(k + 1, int(ratio * n))
        idx = rng.choice(n, size=sz, replace=False)

        # Assign sample to nearest centroid
        diff  = Z[idx, None, :] - centroids[None, :, :]
        boot  = np.sqrt((diff**2).sum(axis=2)).argmin(axis=1)
        orig  = labels[idx]

        # Match clusters (Hungarian)
        overlap = np.zeros((k, k), int)
        for a, b in zip(orig, boot):
            if 0 <= a < k and 0 <= b < k:
                overlap[a, b] += 1
        ri, ci = linear_sum_assignment(-overlap)
        mapping = dict(zip(ri, ci))

        for c in range(k):
            A = set(np.where(orig == c)[0])
            B = set(np.where(boot == mapping.get(c, c))[0])
            inter = len(A & B)
            union = len(A | B)
            scores[run, c] = inter / union if union else 0.0

    return scores.mean(axis=0), float(scores.mean())

## 8. k-search

Build DTW once on a sample, run consensus + all three metrics for k=2..12.
Prints a ranked table and plots all three curves.
Lower Davies-Bouldin = better. Higher Silhouette and Jaccard = better.
The recommended k is highlighted — it maximises a simple combined rank score.


In [ ]:
def ksearch(Z_full, ids_full, mask, name, k_range=range(K_MIN, K_MAX+1),
            sample_n=KSEARCH_SAMPLE, seed=RANDOM_STATE):
    rng  = np.random.default_rng(seed)
    Z_active = Z_full[mask]
    n_active = len(Z_active)

    # Sample for DTW matrix
    if sample_n is None or sample_n >= n_active:
        idx = np.arange(n_active)
    else:
        idx = np.sort(rng.choice(n_active, size=sample_n, replace=False))
    Z_sample = Z_active[idx]

    print(f"\n{'='*55}")
    print(f"k-search: {name}  (sample={len(Z_sample)}, active={n_active})")
    print('='*55)
    print("Building DTW matrix...")
    D = build_dtw_matrix(Z_sample)

    rows = []
    for k in k_range:
        t0  = time.time()
        sim = consensus(D, k)
        lbl = cluster_from_sim(sim, k)

        # Assign full active set via Euclidean to sample centroids
        centroids = np.array([Z_sample[lbl == c].mean(axis=0)
                               for c in range(k)])
        diff      = Z_active[:, None, :] - centroids[None, :, :]
        full_lbl  = np.sqrt((diff**2).sum(axis=2)).argmin(axis=1)

        s  = sil(Z_active, full_lbl)
        d  = dbi(Z_active, full_lbl)
        _, j = jaccard_stability(Z_active, full_lbl, k, n_boot=5)

        elapsed = time.time() - t0
        sizes   = np.bincount(full_lbl, minlength=k).tolist()
        rows.append(dict(k=k, silhouette=s, davies_bouldin=d, jaccard=j,
                         time_s=round(elapsed,1), sizes=sizes))
        print(f"  k={k:2d}  sil={s:.3f}  DB={d:.3f}  jac={j:.3f}  {elapsed:.0f}s  sizes={sizes}")

    df = pd.DataFrame(rows)

    # Combined rank: higher sil rank + lower DB rank + higher jac rank
    df["rank_sil"] = df["silhouette"].rank(ascending=False)
    df["rank_db"]  = df["davies_bouldin"].rank(ascending=True)
    df["rank_jac"] = df["jaccard"].rank(ascending=False)
    df["combined_rank"] = df["rank_sil"] + df["rank_db"] + df["rank_jac"]
    best_k = int(df.loc[df["combined_rank"].idxmin(), "k"])

    print(f"\nRecommended k = {best_k}  (lowest combined rank across all three metrics)")
    return df, best_k


results = {}

df_raw, best_k_raw = ksearch(Z_raw, ids_raw, mask_raw, "raw")
results["raw"] = (df_raw, best_k_raw)

df_znorm, best_k_znorm = ksearch(Z_znorm, ids_znorm, mask_znorm, "znorm")
results["znorm"] = (df_znorm, best_k_znorm)

## 9. k-search plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False)
metrics = [
    ("silhouette",    "Silhouette",    True),   # True = higher is better
    ("davies_bouldin","Davies-Bouldin",False),
    ("jaccard",       "Jaccard",       True),
]
colors = {"raw": "steelblue", "znorm": "darkorange"}

for row_i, (dname, (df, best_k)) in enumerate(results.items()):
    for col_i, (col, label, higher_better) in enumerate(metrics):
        ax = axes[row_i, col_i]
        ax.plot(df["k"], df[col], marker="o", color=colors[dname], linewidth=1.5)
        ax.axvline(best_k, color="crimson", linestyle="--", linewidth=1, label=f"best k={best_k}")
        ax.set_title(f"{dname} — {label}", fontsize=10)
        ax.set_xlabel("k")
        ax.set_ylabel(label)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        arrow = "↑ better" if higher_better else "↓ better"
        ax.text(0.97, 0.05, arrow, transform=ax.transAxes,
                ha="right", va="bottom", fontsize=8, color="gray")

plt.suptitle("k-search: raw vs z-norm", fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / "ksearch.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved → {OUT_DIR / 'ksearch.png'}")

print("\nSummary table — raw:")
print(results["raw"][0][["k","silhouette","davies_bouldin","jaccard","combined_rank"]].to_string(index=False))
print("\nSummary table — znorm:")
print(results["znorm"][0][["k","silhouette","davies_bouldin","jaccard","combined_rank"]].to_string(index=False))

## 10. Choose k

Edit `CHOSEN_K` and `CHOSEN_DATA` if you want to override the recommendation.
The recommended values come from the k-search above.


In [ ]:
# Change these if the recommendation doesn't match your visual inspection
CHOSEN_K    = best_k_raw   # or best_k_znorm, or any int
CHOSEN_DATA = "raw"        # "raw" or "znorm"

print(f"Will cluster: {CHOSEN_DATA}  k={CHOSEN_K}")
print(f"Recommended: raw={best_k_raw}  znorm={best_k_znorm}")

## 11. Full clustering with chosen k

In [ ]:
Z_full  = Z_raw   if CHOSEN_DATA == "raw" else Z_znorm
mask    = mask_raw if CHOSEN_DATA == "raw" else mask_znorm
ids     = ids_raw  if CHOSEN_DATA == "raw" else ids_znorm
Z_active = Z_full[mask]
n_flat   = (~mask).sum()

print(f"Data: {CHOSEN_DATA}  k={CHOSEN_K}")
print(f"Active series: {len(Z_active)}  |  flat (excluded from DTW): {n_flat}")

if FINAL_SAMPLE is not None and FINAL_SAMPLE < len(Z_active):
    rng = np.random.default_rng(RANDOM_STATE)
    idx = np.sort(rng.choice(len(Z_active), size=FINAL_SAMPLE, replace=False))
    Z_fit = Z_active[idx]
    print(f"Using sample of {FINAL_SAMPLE} for DTW matrix")
else:
    Z_fit = Z_active
    idx   = np.arange(len(Z_active))
    print("Using all active series for DTW matrix")

print("Building full DTW matrix...")
D_full = build_dtw_matrix(Z_fit)

print(f"Running consensus (k={CHOSEN_K}, {CONSENSUS_RUNS} runs)...")
sim_full   = consensus(D_full, CHOSEN_K)
lbl_sample = cluster_from_sim(sim_full, CHOSEN_K)

# Medoid vectors in z-norm space
mpos     = medoid_positions(D_full, lbl_sample, CHOSEN_K)
medoids  = Z_fit[mpos]

# Assign ALL series (including flat ones) to nearest medoid
diff       = Z_full[:, None, :] - medoids[None, :, :]
all_labels = np.sqrt((diff**2).sum(axis=2)).argmin(axis=1)

sizes = np.bincount(all_labels, minlength=CHOSEN_K).tolist()
print(f"\nCluster sizes (all {len(all_labels)} series): {sizes}")

## 12. Validation

In [ ]:
Z_active_full = Z_full[mask]
active_labels = all_labels[mask]

s = sil(Z_active_full, active_labels)
d = dbi(Z_active_full, active_labels)
jac_per, jac_mean = jaccard_stability(Z_active_full, active_labels, CHOSEN_K)

print(f"Final validation  (k={CHOSEN_K}, {CHOSEN_DATA})")
print(f"  Silhouette    : {s:.4f}  {'(strong)' if s>0.5 else '(reasonable)' if s>0.2 else '(weak)'}")
print(f"  Davies-Bouldin: {d:.4f}  {'(good)' if d<1.0 else '(borderline)' if d<2.0 else '(poor)'}")
print(f"  Jaccard (mean): {jac_mean:.4f}  {'(stable)' if jac_mean>0.6 else '(borderline)' if jac_mean>0.4 else '(unstable)'}")
print()
print(f"  Per-cluster Jaccard:")
for c, j in enumerate(jac_per):
    n = int((all_labels == c).sum())
    label = "stable" if j>0.6 else "borderline" if j>0.4 else "unstable"
    print(f"    Cluster {c+1}: {j:.3f}  ({label})  n={n}")

## 13. Medoid plots

In [ ]:
k = CHOSEN_K
fig, axes = plt.subplots(k, 1, figsize=(12, 3*k), sharex=True)
if k == 1:
    axes = [axes]
colors = plt.cm.tab10(np.linspace(0, 1, k))

for i in range(k):
    n_members = int((all_labels == i).sum())
    axes[i].plot(medoids[i], color=colors[i], linewidth=2, label="medoid")
    axes[i].set_title(f"Cluster {i+1}  (n={n_members})", fontsize=10)
    axes[i].set_ylabel("z-norm")
    axes[i].legend(fontsize=8)
    axes[i].grid(True, alpha=0.25)

plt.xlabel("Day of year")
plt.suptitle(f"Medoids — {CHOSEN_DATA}, k={k}", fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / f"medoids_{CHOSEN_DATA}_k{k}.png", dpi=120, bbox_inches="tight")
plt.show()

## 14. Export

In [ ]:
out_df = pd.DataFrame({"ID": ids, "cluster": all_labels})
fname  = OUT_DIR / f"kmedoid_{CHOSEN_DATA}_k{CHOSEN_K}.csv"
out_df.to_csv(fname, index=False)
print(f"Saved: {fname}")
print(f"Cluster sizes: {dict(zip(*np.unique(all_labels, return_counts=True)))}")